In [1]:
import os
import numpy as np
import pandas as pd
import random
import scipy.io as sio
from scipy.io import loadmat
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
csv_filename = "records.csv"
csv_path = os.path.join(parent_dir, csv_filename)

BASE_DIR = "UTD-MHAD Dataset-20251013T122259Z-1-001"
DATASET_DIR = os.path.join(BASE_DIR, "UTD-MHAD Dataset")

## UTD-MHAD Dataset

In [2]:
def load_utd_mhad_dataset():
    SUBFOLDERS = ["depth", "inertial", "skeleton"]
    
    all_files = [f for f in os.listdir(os.path.join(DATASET_DIR, "depth")) if f.endswith(".mat")]
    subjects = sorted(list({f.split("_")[1] for f in all_files}))
    print("Subjects:", subjects)
    
    all_samples = []
    
    for file in all_files:
        subj = file.split("_")[1]
        
        # Parse filename: e.g., "a1_s1_t1_depth.mat"
        base_name_with_modality = file.replace(".mat", "")
        parts = base_name_with_modality.split("_")
        sample_prefix = "_".join(parts[:-1])  # Remove modality suffix
        
        # Check that corresponding mat files exist in all modalities
        exists_in_all = all(
            os.path.exists(os.path.join(DATASET_DIR, sub, sample_prefix + "_" + sub + ".mat"))
            for sub in SUBFOLDERS
        )
        
        if not exists_in_all:
            print(f"Skipping incomplete sample: {sample_prefix}")
            continue
        
        label = file.split("_")[0]
        all_samples.append({"subject": subj, "label": label, "basename": sample_prefix})
    
    print(f"Total samples loaded metadata: {len(all_samples)}")


    def load_sample(sample):
        """Load one sample's depth, inertial, and skeleton data."""
        data = {}
        for sub in SUBFOLDERS:
            fpath = os.path.join(DATASET_DIR, sub, sample["basename"] + "_" + sub + ".mat")
            try:
                mat = loadmat(fpath)
                # Extract first key not starting with '__'
                key = next(k for k in mat.keys() if not k.startswith("__"))
                data[sub + "_mat"] = mat[key]
            except Exception as e:
                print(f"Error reading {fpath}: {e}")
                return None
        
        data["label"] = sample["label"]
        data["subject"] = sample["subject"]
        return data
    
    # Load all samples
    all_data = []
    for s in tqdm(all_samples, desc="Loading all samples"):
        d = load_sample(s)
        if d is not None:
            all_data.append(d)
    
    print(f"Total samples loaded: {len(all_data)}")
    return all_data


X = load_utd_mhad_dataset()

Subjects: ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8']
Total samples loaded metadata: 861


Loading all samples: 100%|██████████| 861/861 [00:17<00:00, 50.39it/s]

Total samples loaded: 861


In [3]:
sample = X[0]
print("\nSample keys:", sample.keys())
print("Depth shape:", sample["depth_mat"].shape)
print("Inertial shape:", sample["inertial_mat"].shape)
print("Skeleton shape:", sample["skeleton_mat"].shape)
print("Label:", sample["label"])
print("Subject:", sample["subject"])


Sample keys: dict_keys(['depth_mat', 'inertial_mat', 'skeleton_mat', 'label', 'subject'])
Depth shape: (240, 320, 55)
Inertial shape: (157, 6)
Skeleton shape: (20, 3, 55)
Label: a2
Subject: s5


## Structured Gaussian Noise (25% of units, 20% std strength)

In [4]:
import numpy as np

NOISE_FRACTION = 0.25   # fraction of structured units to corrupt
NOISE_STRENGTH = 0.20   # noise std = this × unit's own std
SEED = 42
rng = np.random.RandomState(SEED)

def add_noise_skeleton_joints(skeleton_mat, fraction=NOISE_FRACTION, strength=NOISE_STRENGTH):
    """
    skeleton_mat: (20 joints, 3 coords, frames)
    Add Gaussian noise to `fraction` of joints.
    Noise std per joint = strength × std of that joint's data.
    """
    skeleton_mat = skeleton_mat.astype(np.float64).copy()
    n_joints = skeleton_mat.shape[0]
    n_noisy = max(1, int(round(n_joints * fraction)))
    joints_to_noise = rng.choice(n_joints, size=n_noisy, replace=False)
    for j in joints_to_noise:
        joint_data = skeleton_mat[j]          # (3, frames)
        sigma = joint_data.std() * strength
        noise = rng.normal(0, sigma, size=joint_data.shape)
        skeleton_mat[j] += noise
    return skeleton_mat

def add_noise_inertial_channels(inertial_mat, fraction=NOISE_FRACTION, strength=NOISE_STRENGTH):
    """
    inertial_mat: (time_steps, 6 channels)
    Add Gaussian noise to `fraction` of channels.
    Noise std per channel = strength × std of that channel's data.
    """
    inertial_mat = inertial_mat.astype(np.float64).copy()
    n_channels = inertial_mat.shape[1]
    n_noisy = max(1, int(round(n_channels * fraction)))
    channels_to_noise = rng.choice(n_channels, size=n_noisy, replace=False)
    for ch in channels_to_noise:
        ch_data = inertial_mat[:, ch]          # (time_steps,)
        sigma = ch_data.std() * strength
        noise = rng.normal(0, sigma, size=ch_data.shape)
        inertial_mat[:, ch] += noise
    return inertial_mat

def add_noise_depth_frames(depth_mat, fraction=NOISE_FRACTION, strength=NOISE_STRENGTH):
    """
    depth_mat: (H, W, frames)
    Add Gaussian noise to `fraction` of frames.
    Noise std per frame = strength × std of that frame's pixel values.
    """
    depth_mat = depth_mat.astype(np.float64).copy()
    n_frames = depth_mat.shape[2]
    n_noisy = max(1, int(round(n_frames * fraction)))
    frames_to_noise = rng.choice(n_frames, size=n_noisy, replace=False)
    for f in frames_to_noise:
        frame_data = depth_mat[:, :, f]        # (H, W)
        sigma = frame_data.std() * strength
        noise = rng.normal(0, sigma, size=frame_data.shape)
        depth_mat[:, :, f] += noise
    return depth_mat

# Apply structured noise to all samples
for sample in tqdm(X, desc="Applying structured Gaussian noise"):
    sample["skeleton_mat"] = add_noise_skeleton_joints(sample["skeleton_mat"])
    sample["inertial_mat"] = add_noise_inertial_channels(sample["inertial_mat"])
    sample["depth_mat"]    = add_noise_depth_frames(sample["depth_mat"])

# Verify
s = X[0]
print("After structured Gaussian noise injection:")
print(f"  Skeleton:  {s['skeleton_mat'].shape}  — "
      f"{max(1,int(round(s['skeleton_mat'].shape[0]*NOISE_FRACTION)))}/{s['skeleton_mat'].shape[0]} joints noised")
print(f"  Inertial:  {s['inertial_mat'].shape}  — "
      f"{max(1,int(round(s['inertial_mat'].shape[1]*NOISE_FRACTION)))}/{s['inertial_mat'].shape[1]} channels noised")
print(f"  Depth:     {s['depth_mat'].shape}  — "
      f"{max(1,int(round(s['depth_mat'].shape[2]*NOISE_FRACTION)))}/{s['depth_mat'].shape[2]} frames noised")

Applying structured Gaussian noise: 100%|██████████| 861/861 [00:37<00:00, 22.93it/s]

After structured Gaussian noise injection:
  Skeleton:  (20, 3, 55)  — 5/20 joints noised
  Inertial:  (157, 6)  — 2/6 channels noised
  Depth:     (240, 320, 55)  — 14/55 frames noised


## Feature Extraction

In [5]:
def extract_features_from_depth(depth):
    """
    depth: np.array of shape (H, W, frames) => (240, 320, 55)
    We transpose to (frames, H, W) before extraction.
    Returns: 1D feature vector
    """
    depth = np.transpose(depth, (2, 0, 1))  # (frames, H, W)
    features = []

    flattened = depth.reshape(depth.shape[0], -1)  # (frames, H*W)

    # Temporal stats per frame
    frame_mean = flattened.mean(axis=1)
    frame_std = flattened.std(axis=1)
    frame_min = flattened.min(axis=1)
    frame_max = flattened.max(axis=1)
    
    # Aggregate over time
    features.extend([frame_mean.mean(), frame_mean.std()])
    features.extend([frame_std.mean(), frame_std.std()])
    features.extend([frame_min.mean(), frame_min.std()])
    features.extend([frame_max.mean(), frame_max.std()])
    
    # Histogram features over all pixels
    hist, _ = np.histogram(flattened, bins=20)
    features.extend(hist / hist.sum())  # normalize

    # Percentiles over all pixel intensities and frames
    features.extend(np.percentile(flattened, [10, 25, 50, 75, 90]))

    # Temporal differences (motion)
    diff = np.diff(flattened, axis=0)
    features.extend([diff.mean(), diff.std(), diff.max(), diff.min()])

    return np.array(features)

In [6]:
def flatten_numeric(x):
    flat = []
    if isinstance(x, (list, tuple, np.ndarray)):
        for item in x:
            flat.extend(flatten_numeric(item))
    else:
        flat.append(float(x))
    return flat

In [7]:
def extract_features_from_sensor(sensor):
    """
    sensor shape: (time_steps=157, channels=6)
    Extract stats per channel
    """
    sensor = np.array(sensor)
    features = []
    for ch in range(sensor.shape[1]):
        channel_data = sensor[:, ch]
        features.append(channel_data.mean())
        features.append(channel_data.std())
        features.append(channel_data.min())
        features.append(channel_data.max())
        features.extend(np.percentile(channel_data, [10, 25, 50, 75, 90]))
    return np.array(features)



In [8]:
def extract_features_from_skeleton(skeleton):
    """
    skeleton: np.array of shape (20 joints, 3 coords, 55 frames)
    Extracts stats across frames and joints by reshaping.
    """
    skeleton = np.array(skeleton)

    # Reshape to (frames, joints*coords)
    reshaped = skeleton.transpose(2, 0, 1).reshape(skeleton.shape[2], -1)  # (55, 60)

    features = []
    features.extend(reshaped.mean(axis=0))  # mean per feature across frames
    features.extend(reshaped.std(axis=0))
    features.extend(reshaped.min(axis=0))
    features.extend(reshaped.max(axis=0))

    return np.array(features)

In [9]:
# Note: NOISE_FRACTION, NOISE_STRENGTH, and rng are defined in the structured noise cell above

import mediapipe as mp
from tqdm import tqdm
import cv2

mp_pose = mp.solutions.pose


def load_video_frames(video_path, resize_shape=None):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if resize_shape is not None:
            frame = cv2.resize(frame, resize_shape)
        frames.append(frame)

    cap.release()

    return np.array(frames)  # (num_frames, H, W, C)


def extract_video_basic_features(frames):
    # Convert to grayscale if frames are RGB
    if frames.ndim == 4 and frames.shape[-1] == 3:
        frames_gray = np.array([cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames])
    else:
        frames_gray = frames
    
    flattened = frames_gray.reshape(frames_gray.shape[0], -1)  # (frames, pixels)
    
    frame_mean = flattened.mean(axis=1)
    frame_std = flattened.std(axis=1)
    
    features = [
        frame_mean.mean(),
        frame_mean.std(),
        frame_std.mean(),
        frame_std.std(),
        flattened.min(),
        flattened.max()
    ]

    return np.array(features)


def extract_pose_features(frames, resize_shape=(320, 240)):
    """
    frames: np.array of shape (num_frames, H, W, C) in BGR format (OpenCV style)
    Returns: pose feature vector of shape (264,)
    """
    pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5)
    keypoints_all_frames = []

    for frame in frames:
        resized_frame = cv2.resize(frame, resize_shape)
        frame_rgb = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2RGB)
        result = pose.process(frame_rgb)

        if result.pose_landmarks:
            landmarks = result.pose_landmarks.landmark
            coords = []

            for lm in landmarks:
                coords.extend([lm.x, lm.y])  # normalized landmark x, y

            keypoints_all_frames.append(coords)
        else:
            keypoints_all_frames.append([0]*(33*2))  # zero vector if no pose detected

    pose.close()

    if not keypoints_all_frames:
        return np.zeros(33*2*4)

    keypoints_all_frames = np.array(keypoints_all_frames)  # (num_frames, 66)

    features = []
    features.extend(keypoints_all_frames.mean(axis=0))
    features.extend(keypoints_all_frames.std(axis=0))
    features.extend(keypoints_all_frames.min(axis=0))
    features.extend(keypoints_all_frames.max(axis=0))

    return np.array(features)


def load_utd_mhad_videos(DATASET_DIR):
    """
    Load all UTD-MHAD video samples from RGB folders.
    Returns all samples without train/test split.
    """
    VIDEO_SUBFOLDERS = ['RGB-part1', 'RGB-part2', 'RGB-part3', 'RGB-part4']

    # Collect all video files from all RGB subfolders
    all_files = []
    for subfolder in VIDEO_SUBFOLDERS:
        folder_path = os.path.join(DATASET_DIR, subfolder)
        if not os.path.exists(folder_path):
            print(f"Warning: {folder_path} does not exist, skipping...")
            continue
        files = [os.path.join(subfolder, f) for f in os.listdir(folder_path) if f.endswith('.avi')]
        all_files.extend(files)

    subjects = sorted(list({os.path.basename(f).split('_')[1] for f in all_files}))
    print("All Subjects:", subjects)

    all_samples = []

    for f in all_files:
        filename = os.path.basename(f)
        subj = filename.split('_')[1] 
        label = filename.split('_')[0]

        sample_prefix = filename.replace('.avi', '')
        all_samples.append({"subject": subj, "label": label, "filepath": f, "basename": sample_prefix})

    print(f"Total video samples: {len(all_samples)}")

    # Load and extract features from all samples
    def load_sample(sample):
        """Load and extract features from one video sample."""
        video_path = os.path.join(DATASET_DIR, sample["filepath"])
        try:
            frames = load_video_frames(video_path, resize_shape=(120, 160))
            
            # --- Structured noise: add Gaussian noise to 10% of video frames ---
            frames = frames.astype(np.float64)
            n_frames = frames.shape[0]
            n_noisy = max(1, int(round(n_frames * NOISE_FRACTION)))
            frames_to_noise = rng.choice(n_frames, size=n_noisy, replace=False)
            for fi in frames_to_noise:
                frame_data = frames[fi]
                sigma = frame_data.std() * NOISE_STRENGTH
                noise = rng.normal(0, sigma, size=frame_data.shape)
                frames[fi] += noise
            frames = np.clip(frames, 0, 255).astype(np.uint8)
            # ---
            
            video_feat = extract_pose_features(frames)
            
            return {
                "video_feat": video_feat,
                "label": sample['label'],
                "subject": sample['subject']
            }
        except Exception as e:
            print(f"Error processing {video_path}: {e}")
            return None

    # Load all video samples
    all_data = []
    for sample in tqdm(all_samples, desc="Loading video samples"):
        data = load_sample(sample)
        if data is not None:
            all_data.append(data)

    print(f"Total video samples loaded: {len(all_data)}")
    return all_data

In [10]:
X_feat, y, subjects = [], [], []

for sample in tqdm(X, desc="Extracting features"):
    depth_feat = extract_features_from_depth(sample["depth_mat"])
    sensor_feat = extract_features_from_sensor(sample["inertial_mat"])
    skeleton_feat = extract_features_from_skeleton(sample["skeleton_mat"])
    
    X_feat.append({
        "depth_feat": depth_feat,
        "sensor_feat": sensor_feat,
        "skeleton_feat": skeleton_feat
    })
    y.append(sample["label"])
    subjects.append(sample["subject"])

print("\nFeature extraction complete!")
print("Depth Features:", X_feat[0]["depth_feat"].shape)
print("Sensor Features:", X_feat[0]["sensor_feat"].shape)
print("Skeleton Features:", X_feat[0]["skeleton_feat"].shape)

print("\nDataset size:", len(X_feat))
print("\nFeature dimensions:")
print("Depth:", X_feat[0]["depth_feat"].shape)
print("Sensor:", X_feat[0]["sensor_feat"].shape)
print("Skeleton:", X_feat[0]["skeleton_feat"].shape)

Extracting features: 100%|██████████| 861/861 [01:35<00:00,  9.00it/s]


Feature extraction complete!
Depth Features: (37,)
Sensor Features: (54,)
Skeleton Features: (240,)

Dataset size: 861

Feature dimensions:
Depth: (37,)
Sensor: (54,)
Skeleton: (240,)


In [11]:
X_videos = load_utd_mhad_videos(DATASET_DIR)

# Check sample structure
if len(X_videos) > 0:
    print("\nSample keys:", X_videos[0].keys())
    print("Video features shape:", X_videos[0]["video_feat"].shape)
    print("Label:", X_videos[0]["label"])
    print("Subject:", X_videos[0]["subject"])


# Extract features and labels
video_features = []
video_labels = []
video_subjects = []

for sample in X_videos:
    video_features.append(sample["video_feat"])
    video_labels.append(sample["label"])
    video_subjects.append(sample["subject"])

print("\nVideo features extracted:")
print("Features shape:", np.array(video_features).shape)
print("Number of samples:", len(video_features))

All Subjects: ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8']
Total video samples: 861


Loading video samples:   0%|          | 0/861 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1772714165.165268 1920652 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1772714165.234627 1921573 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 590.48.01), renderer: NVIDIA GeForce RTX 4090/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1772714165.270825 1921544 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1772714165.294461 1921567 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1772714165.309839 1921541 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_D

Total video samples loaded: 861

Sample keys: dict_keys(['video_feat', 'label', 'subject'])
Video features shape: (264,)
Label: a3
Subject: s1

Video features extracted:
Features shape: (861, 264)
Number of samples: 861


In [12]:
video_lookup = {}
for v_sample in X_videos:
    key = (v_sample['subject'], v_sample['label'])
    video_lookup[key] = v_sample['video_feat']

aligned_video_features = []
for i, sample in enumerate(X):
    key = (sample['subject'], sample['label'])
    
    if key in video_lookup:
        aligned_video_features.append(video_lookup[key])
    else:
        # If no video found, use zeros or skip this sample
        print(f"Warning: No video for subject {sample['subject']}, label {sample['label']}")
        aligned_video_features.append(np.zeros(264))  # or skip

In [13]:
for i, sample in enumerate(X_feat):
    sample["video_feat"] = aligned_video_features[i]

In [14]:
X_feat[0]

{'depth_feat': array([ 2.48738846e+02,  7.66788230e+00,  8.02776366e+02,  1.52679527e+01,
        -1.77222819e+02,  3.04125952e+02,  3.21738500e+03,  1.43163874e+02,
         5.94223485e-05,  3.73366477e-03,  4.72071496e-02,  7.96426136e-01,
         5.82244318e-02,  5.76420455e-03,  9.77746212e-05,  2.36742424e-07,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  2.36742424e-07,
         1.72821970e-05,  3.32859848e-04,  1.83712121e-03,  6.52580492e-03,
         6.88070549e-02,  1.02665720e-02,  6.77556818e-04,  2.24905303e-05,
        -2.76524237e+01,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         2.63019493e+02, -1.21221547e-02,  1.95153698e+02,  3.60266171e+03,
        -3.48734264e+03]),
 'sensor_feat': array([-6.93620662e-01,  3.47750448e-01, -1.31543000e+00, -7.08010000e-02,
        -1.05366200e+00, -9.95850000e-01, -6.98730000e-01, -3.77686000e-01,
        -2.05273200e-01, -2.36928357e-01,  5.85372208e-01, -1.78393600e+00,
         1.05004900e+00, -9.6499

In [15]:
le = LabelEncoder()
le.fit(y)
y_enc = le.transform(y)
print("\nEncoded:", y_enc.shape)
print("Classes:", le.classes_)


Encoded: (861,)
Classes: ['a1' 'a10' 'a11' 'a12' 'a13' 'a14' 'a15' 'a16' 'a17' 'a18' 'a19' 'a2'
 'a20' 'a21' 'a22' 'a23' 'a24' 'a25' 'a26' 'a27' 'a3' 'a4' 'a5' 'a6' 'a7'
 'a8' 'a9']


In [16]:
import joblib

SAVE_DIR = "features_25_noise"
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(X_feat, os.path.join(SAVE_DIR, "X_feat.pkl"), compress=3)
print("\nSaved feature dictionary")

np.save(os.path.join(SAVE_DIR, "y.npy"), y_enc)
print("Saved encoded labels")

np.save(os.path.join(SAVE_DIR, "subjects.npy"), subjects)
print("Saved subjects")

joblib.dump(le, os.path.join(SAVE_DIR, "label_encoder.pkl"))
print("Saved LabelEncoder")


Saved feature dictionary
Saved encoded labels
Saved subjects
Saved LabelEncoder


In [17]:
import joblib

X_feat = joblib.load("features_25_noise/X_feat.pkl")
y = np.load("features_25_noise/y.npy")
subjects = np.load("features_25_noise/subjects.npy")

le = joblib.load("features_25_noise/label_encoder.pkl")
print("Classes:", le.classes_)

Classes: ['a1' 'a10' 'a11' 'a12' 'a13' 'a14' 'a15' 'a16' 'a17' 'a18' 'a19' 'a2'
 'a20' 'a21' 'a22' 'a23' 'a24' 'a25' 'a26' 'a27' 'a3' 'a4' 'a5' 'a6' 'a7'
 'a8' 'a9']


In [18]:
X_feat[0]

{'depth_feat': array([ 2.48738846e+02,  7.66788230e+00,  8.02776366e+02,  1.52679527e+01,
        -1.77222819e+02,  3.04125952e+02,  3.21738500e+03,  1.43163874e+02,
         5.94223485e-05,  3.73366477e-03,  4.72071496e-02,  7.96426136e-01,
         5.82244318e-02,  5.76420455e-03,  9.77746212e-05,  2.36742424e-07,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  2.36742424e-07,
         1.72821970e-05,  3.32859848e-04,  1.83712121e-03,  6.52580492e-03,
         6.88070549e-02,  1.02665720e-02,  6.77556818e-04,  2.24905303e-05,
        -2.76524237e+01,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         2.63019493e+02, -1.21221547e-02,  1.95153698e+02,  3.60266171e+03,
        -3.48734264e+03]),
 'sensor_feat': array([-6.93620662e-01,  3.47750448e-01, -1.31543000e+00, -7.08010000e-02,
        -1.05366200e+00, -9.95850000e-01, -6.98730000e-01, -3.77686000e-01,
        -2.05273200e-01, -2.36928357e-01,  5.85372208e-01, -1.78393600e+00,
         1.05004900e+00, -9.6499

In [19]:
print("Samples:", len(X_feat))
print("Depth feat shape:", X_feat[0]["depth_feat"].shape)
print("Sensor feat shape:", X_feat[0]["sensor_feat"].shape)
print("Skeleton feat shape:", X_feat[0]["skeleton_feat"].shape)
print("Video feat shape:", X_feat[0]["video_feat"].shape)

Samples: 861
Depth feat shape: (37,)
Sensor feat shape: (54,)
Skeleton feat shape: (240,)
Video feat shape: (264,)


## Data preprocessing

In [20]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.preprocessing import StandardScaler
import numpy as np

X_depth = np.stack([x["depth_feat"] for x in X_feat])
X_sensor = np.stack([x["sensor_feat"] for x in X_feat])
X_skel = np.stack([x["skeleton_feat"] for x in X_feat])
X_video = np.stack([x["video_feat"] for x in X_feat])


scaler_depth = StandardScaler()
X_depth = scaler_depth.fit_transform(X_depth)

scaler_sensor = StandardScaler()
X_sensor = scaler_sensor.fit_transform(X_sensor)

scaler_skel = StandardScaler()
X_skel = scaler_skel.fit_transform(X_skel)

scaler_video = StandardScaler()
X_video = scaler_video.fit_transform(X_video)

print("Feature shapes:")
print(f"Depth: {X_depth.shape}")    # (861, 37)
print(f"Sensor: {X_sensor.shape}")  # (861, 54)
print(f"Skeleton: {X_skel.shape}")  # (861, 240)
print(f"Video: {X_video.shape}")    # (861, 264)

Feature shapes:
Depth: (861, 37)
Sensor: (861, 54)
Skeleton: (861, 240)
Video: (861, 264)
